In [1]:
import pandas as pd
import altair as alt
import numpy as np
from altair_saver import save
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

# Bracken

## Bracken raw df

In [ ]:
def df_bracken_species_raw(file: str, 
                           level: str = 'species',
                           cutoff: float = 0.001,
                           virus_only: bool = True) -> pd.DataFrame:
    '''
    Returns a df for the taxonomy found in the cleaned raw bracken report.
    Used to generate bar plots of the different taxonomies. 
    
    :param str file: Path to the bracken report.
    :param str level: Level of taxonomy. [domain, phylum, class, order, family, genus, species]. Default = 'species'
    :param float cutoff: Cutoff of percent the taxonomy level is present in. Default = 0.05
    :param bool virus_only: Only include Viruses. Default = True
    :return: pd.DataFrame
    '''
    
    taxonomy = {'domain': 'D', 
                'phylum': 'P', 
                'class': 'K', 
                'order': 'O',
                'family': 'F',
                'genus': 'G',
                'species': 'S'}
     
    df = (
            pd.read_csv(file)
              .loc[lambda x: x.level == taxonomy[level]]
              .loc[lambda x: x.percent > cutoff]
              .sort_values('percent', ascending=False)
               
           )
    
    if virus_only:
        return df.loc[lambda x: x.domain == 'Virus']
    return df


## Bracken bar plot

In [ ]:
def bar_chart_bracken_raw(file: str, 
                          level: str = 'species',
                          cutoff: float = 0.05,
                          number: int = 10,
                          virus_only=True) -> altair.vegalite.v4.api.Chart:
    '''
    Returns a bar chart of the taxnomies from the bracken species file in the cleaned_files folder. 
    
    :param str file: Path to the cleaned bracken report in the cleaned_files folder.
    :param str level: Level of taxonomy. [domain, phylum, class, order, family, genus, species]. Default = 'species'
    :param float cutoff: Cutoff of percent the taxonomy level is present in. Default = 0.05
    :param int number: The number bars to plot. Default = 10
    :return: altair.vegalite.v4.api.Chart
    '''
    
    df = df_bracken_species_raw(file, level, cutoff, virus_only)
    
    return alt.Chart(df.head(number), title='Bracken classification raw').mark_bar().encode(
     alt.X('percent:Q', axis=alt.Axis(format='.1%'), title='Percent of reads'),
     alt.Y('name:N', sort='-x', title=None),
     alt.Color('name:N', title=None),
     tooltip=[alt.Tooltip('domain:N'), alt.Tooltip('percent:Q', format='.1%')]
      ).properties(
        width=500, height=500)
  

plot = bar_chart_bracken_raw('results/cleaned_files/Bat-Guano-15_S6_L001_R_bracken_raw.csv', 
                             'species',
                             cutoff=0.001,
                             number=6,
                             virus_only=True)

plot

## Bracken pie chart of Kingdom abundance:

In [ ]:
def pie_chart_bracken_raw(file: str):
    
    df = df_bracken_species_raw(file, 
                                virus_only=False,
                                level='domain')

    pie = alt.Chart(df, title='Abundance of Kingdoms').mark_arc(innerRadius=50).encode(
     theta=alt.Theta(field="percent", type="quantitative"),
     color=alt.Color(field="name", type="nominal", legend=None),
    ).properties(width=500)
    
    text = pie.mark_text(radius=174, size=10).encode(text="name:N")
    
    return (pie + text)


plot_bracken_pie('results/cleaned_files/Bat-Guano-15_S6_L001_R_bracken_raw.csv')
                      
                     

# Kaiju

In [ ]:
def bar_chart_kaiju_raw(file: str, 
                        cutoff: float = 0.05,
                        number: int = 10) -> altair.vegalite.v4.api.Chart:
    '''
    Returns a bar chart of the taxnomies from the bracken species file. 
    
    :param str file: Path to the bracken report.
    :param str level: Level of taxonomy. [domain, phylum, class, order, family, genus, species]. Default = 'species'
    :param float cutoff: Cutoff of percent the taxonomy level is present in. Default = 0.05
    :param int number: The number bars to plot. Default = 10
    :return: altair.vegalite.v4.api.Chart
    '''
    
    df = (pd.read_csv(file)
          .groupby(['taxon_id', 'percent', 'taxon_name'], as_index=False)
          .agg(taxonomy=('taxonomy', list))
          .sort_values('percent', ascending=False)
          .loc[lambda x: x.percent > cutoff]
         )
    
    
    return alt.Chart(df.head(number), title='Kaiju classification raw').mark_bar().encode(
     alt.X('percent:Q', axis=alt.Axis(format='.1%'), title='Percent of reads'),
     alt.Y('taxon_name:N', sort='-x', title=None),
     alt.Color('taxon_name:N', title=None),
     tooltip=[alt.Tooltip('taxonomy:O'), alt.Tooltip('percent:Q', format='.1%')]
      ).properties(
        width=500, height=500)
   

bar_chart_kaiju_raw('results/cleaned_files/Bat-Guano-15_S6_L001_R_kaiju_raw.csv', cutoff=0.01)

# MEGAHIT contig histogram

In [ ]:
import numpy as np

def megahit_contig_histogram(file: str) -> altair.vegalite.v4.api.Chart:
    '''
    Plots histogram of the contigs from the megahit assembled contigs.
    :param str file: Path to the csv file for the megahit contigs.
    :return: Altair histogram
    '''
    contigs = pd.read_csv(file)
    
    return alt.Chart(contigs, title='Megahit contigs size').mark_bar().encode(
     alt.X('length:Q', bin=alt.Bin(step=500), title='Length (nt)'),
     alt.Y('count(length):Q', title='Number of contigs')
    ).properties(width=500, height=500)
 
    
megahit_contig_histogram('results/megahit/Bat-Guano-15_S6_L001_R.csv')

### Boxplot contig length megahit

In [ ]:
def megahit_contig_boxplot(file: str) -> altair.vegalite.v4.api.Chart:
    '''
    Plots boxplot of the contigs from the megahit assembled contigs.
    :param str file: Path to the csv file for the megahit contigs.
    :return: Altair boxplot
    '''
    contigs = pd.read_csv(file).assign(group='group1')
    
    return alt.Chart(contigs, title='Boxplot of contigs in Megahit').mark_boxplot(extent='min-max').encode(
    alt.X('group:N', axis=alt.Axis(labels=False, title='Contigs', ticks=False)),
    alt.Y('length:Q', title='Length (nt)')).properties(width=500, height=500)

megahit_contig_boxplot('results/megahit/Bat-Guano-15_S6_L001_R.csv')

# Facet wrap of contig coverage

In [8]:

def plot_contig_coverage_facet(file: str) -> list[alt.vegalite.v4.api.Chart]:
    '''
    Returns a list of plots for every contig in the csv file generated from samtools mpileup and the 
    wrangle_contig_info.py script. 
    :params str file: Path to the samtools mpileup file in the cleaned_files folder.
    :returns: List of altair facet wrap with plots in the categories: short, medium and long.
    '''
    plots = []
    
    contigs_coverage = (pd.read_csv(file)
        .assign(category=lambda x: pd.cut(x.length, bins=[1, 500, 2500, np.inf],
                                          labels=['short', 'medium', 'long']))
        .assign(category=lambda x: pd.Categorical(x.category, ['short', 'medium', 'long']))
        .sort_values('category'))

    
    step_size = {'short': 50, 'medium': 150, 'long': 2000}
    
    for contig in contigs_coverage.category.unique():
        
        base = alt.Chart().mark_line().encode(
         alt.X('position:N', axis=alt.Axis(values=np.arange(0, 20000, step_size[contig]))),
         alt.Y('coverage:Q')
            ).properties(width=125, height=125)

        rule = alt.Chart().mark_rule(color='red').encode(
         alt.Y('mean(coverage):Q')
            ).properties(width=125, height=125)

        layered = (alt.layer(base, rule, data=contigs_coverage.loc[lambda x: x.category == contig])
         .encode(alt.Y(title='Coverage'),
                 alt.X(title='Position'))
         .facet('contig:N', columns=6, title=f'Coverage of contigs with {contig} length')
         .resolve_scale(y='independent', x='independent'))
        
        plots.append(layered)
        
    return plots


    
test = plot_contig_coverage_facet('../data/sample1/results/cleaned_files/Bat-Guano-15_S6_L001_R_contigs_coverage_mpileup.csv')

#save(test[2], 'long.pdf')

# Kaiju on megahit

In [ ]:
def plot_kaiju_megahit(file: str) -> altair.vegalite.v4.api.Chart:
    '''
    Plots taxonomy abundancy of the contigs from the megahit assembly. 
    Needs the "kaiju_out" file. 
    :param str file: Path to the csv file for the kaiju out file on the contigs.
    :return: Altair bar chart
    '''

    kaiju_raw = pd.read_csv(file,
                        sep='\t',
                        header=None,
                        usecols=[1, 2, 7],
                        names=['name', 'taxon_id', 'taxonomy'])

    kaiju = (kaiju_raw
     .dropna()
     .assign(taxonomy=lambda x: x.taxonomy.str.split(';').str[:-1])
     .assign(last_level=lambda x: x.taxonomy.str[-1])
     .assign(second_level=lambda x: x.taxonomy.str[-2])
     .assign(third_level=lambda x: x.taxonomy.str[-3])
     .assign(kingdom=lambda x: np.select([x.taxonomy.str[0] != 'cellular organisms'],
                                                   [x.taxonomy.str[0]],
                                                   default=x.taxonomy.str[1]))
     .drop(columns='taxonomy'))
             
    return alt.Chart(kaiju, title='Kaiju classification on MEGAHIT contigs').mark_bar().encode(
     alt.X('count(last_level):Q', title='Number of occurancies',
           axis=alt.Axis(values=np.arange(0, 200, 1), format='.0f')),
     alt.Y('last_level:N', sort='-x', title=None),
     alt.Color('last_level:N', title=None),
     tooltip=[alt.Tooltip('second_level'), alt.Tooltip('third_level')]
    ).properties(width=500, height=500)
     
    


plot_kaiju_megahit('results/kaiju/megahit/Bat-Guano-15_S6_L001_R_names_megahit.out')



## Abundance plot by CAT classification

In [ ]:
def plot_cat_megahit(file: str) -> altair.vegalite.v4.api.Chart:
    '''
    Plots taxonomy abundancy predicted by CAT of the contigs from the megahit assembly. 
    Needs the CAT file. 
    :param str file: Path to the CAT file made on megahit contigs.
    :return: Altair bar chart
    '''
    cat_raw = pd.read_csv(file,
                  sep='\t')

    cat = (cat_raw
     .rename(columns={'# contig': 'name',
                      'species': 'last_level_cat',
                      'genus': 'second_level_cat',
                      'family': 'third_level_cat'})
     .loc[lambda x: x.classification != 'no taxid assigned']
     .loc[lambda x: x['superkingdom'] != 'no support']
     .loc[lambda x: x['phylum'] != 'no support']
     .drop(columns=['classification', 'lineage', 'lineage scores'])
     .assign(kingdom_cat=lambda x: x['superkingdom'].str[:-6])
     .drop(columns='superkingdom')
     .loc[lambda x: x.last_level_cat != 'no support']
    )
    
    return alt.Chart(cat, title='CAT classification on MEGAHIT contigs').mark_bar().encode(
     alt.X('count(last_level_cat):Q', title='Number of occurancies',
           axis=alt.Axis(values=np.arange(0, 200, 1), format='.0f')),
     alt.Y('last_level_cat:N', sort='-x', title=None),
     alt.Color('last_level_cat:N'),
     tooltip=[alt.Tooltip('second_level_cat'), alt.Tooltip('third_level_cat')]
    ).properties(width=500, height=500)

    
plot_cat_megahit('results/cat/CAT_Bat-Guano-15_S6_L001_R_contigs_names.txt')

#### The below code tries to make a tidy format of the taxonomy. 
It can work, however, the order in which the taxonomy is represented. 
It is not the same for every contig, which makes it hard to capture. 

In [ ]:
taxonomy_list = ['kingdom', 'phylum', 'class', 'order', 'family', 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


(kaiju_raw.merge(megahit, on='name')
 .assign(taxonomy=lambda x: x['taxonomy'].str.split(';').str[:-1])
 
 .assign(taxonomy=lambda x: np.select([x['taxonomy'].str[0] != 'cellular organisms'],
                                     [x['taxonomy']],
                                     default=x['taxonomy'].str[1:]))
 .explode('taxonomy')
 .rename(columns={'taxonomy': 'rank'})
 .dropna()
 .assign(taxonomy=lambda x: x.groupby('name')['name'].transform(lambda x: taxonomy_list[:x.count()]))
)


# The merged CAT and Kaiju megahit csv
What can I do with this file?
- Do some BLAST?
- Assembly the genome by downloading reference genome?
- Phylogenetic tree?

In [ ]:
merged = pd.read_csv('results/cleaned_files/Bat-Guano-15_S6_L001_R_cat_kaiju_merged.csv')
(merged
 #.drop(columns=['aa_match', 'sequence', 'taxonomy', 'taxon_id', 'reason', 'phylum', 'class', 'order'])
 #.loc[lambda x: x.classification == 'taxid assigned']
)

